# 兵王问题-SVM-二分类
## 1.数据集的读取与可视化初步

In [8]:
import pandas as pd

col_names=['wx','wy','wwx','wwy','vx','vy','outcome']
df=pd.read_csv('krkopt.data',names=col_names)

print("数据集形状",df.shape)
print("前五行：")
print(df.head())
print("标签分布：")
print(df["outcome"].value_counts())

数据集形状 (28056, 7)
前五行：
  wx  wy wwx  wwy vx  vy outcome
0  a   1   b    3  c   2    draw
1  a   1   c    1  c   2    draw
2  a   1   c    1  d   1    draw
3  a   1   c    1  d   2    draw
4  a   1   c    2  c   1    draw
标签分布：
outcome
fourteen    4553
thirteen    4194
twelve      3597
eleven      2854
draw        2796
fifteen     2166
ten         1985
nine        1712
eight       1433
seven        683
six          592
five         471
sixteen      390
two          246
four         198
three         81
one           78
zero          27
Name: count, dtype: int64


## 2.坐标特征编码与标签二值化

In [10]:
replace_rules={'^a$':1,'^b$':2,'^c$':3,'^d$':4,'^e$':5,'^f$':6,'^g$':7,'^h$':8}
coord_cols=['wx','wy','wwx','wwy','vx','vy']

df.replace(replace_rules,regex=True,inplace=True)
df[coord_cols]=df[coord_cols].astype(int)

df['lable']=df['outcome'].apply(lambda s:1 if s=='draw' else 0)

print(df['lable'].value_counts())

lable
0    25260
1     2796
Name: count, dtype: int64


## 3.Z-score 标准化与数据集切分

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X=df[coord_cols].values
y=df['lable'].values
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

scaler=StandardScaler()
scaler.fit(X_train)
X_train_scaled=scaler.transform(X_train)
X_test_scaled=scaler.transform(X_test)

print("训练集的均值：",X_train_scaled.mean(axis=0).round(2))
print("训练集标准差：",X_train_scaled.std(axis=0).round(2))

训练集的均值： [ 0. -0. -0. -0.  0. -0.]
训练集标准差： [1. 1. 1. 1. 1. 1.]


## 4. SVM的二分类训练与结果分析


In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score,confusion_matrix,classification_report)

clf=SVC(kernel='rbf',C=1.0,gamma='scale',random_state=42)
clf.fit(X_train_scaled,y_train)

y_pred=clf.predict(X_test_scaled)
print("测试集准确率：",accuracy_score(y_test,y_pred))
print("混淆矩阵：")
print(confusion_matrix(y_test,y_pred))
print("分类报告：")
print(classification_report(y_test,y_pred))

测试集准确率： 0.9951888809693514
混淆矩阵：
[[5037   16]
 [  11  548]]
分类报告：
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5053
           1       0.97      0.98      0.98       559

    accuracy                           1.00      5612
   macro avg       0.98      0.99      0.99      5612
weighted avg       1.00      1.00      1.00      5612

